# CUDA 101: The CUDA Kernel

In [18]:
%%bash
nvidia-smi

Wed Oct 14 21:29:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.01              KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:0C:00.0 Off |                  N/A |
|  0%   43C    P8             24W /  350W |    1738MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
%%bash
for file in *.cpp *.hpp *.cu *.cuh *.so *.py threads __pycache__; do
    rm -rf "$file"
done

In [20]:
%%writefile parse_num.hpp
#pragma once
#include <charconv>
#include <cstdlib>
#include <string_view>
#include <system_error>

template <typename T>
int parse_num(T& val, const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  auto [read_end, read_err] = std::from_chars(arg.data(), arg_end, val);
  bool parse_err = read_err != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_err || partial_read) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}


Writing parse_num.hpp


In [21]:
%%writefile check_grid.cuh
#pragma once
#include <cstdlib>
#include <cuda_runtime.h>

inline int check_grid(const dim3& grid_shape, const cudaDeviceProp& prop) {
  if (grid_shape.x == 0 || grid_shape.y == 0 || grid_shape.z == 0) {
    return EXIT_FAILURE;
  }
  if (grid_shape.x > prop.maxGridSize[0] || grid_shape.y > prop.maxGridSize[1] || grid_shape.z > prop.maxGridSize[2]) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing check_grid.cuh


In [22]:
%%writefile check_block.cuh
#pragma once
#include <cstdlib>
#include <cuda_runtime.h>

inline int check_block(const dim3& block_shape, const cudaDeviceProp& prop) {
  if (block_shape.x == 0 || block_shape.y == 0 || block_shape.z == 0) {
    return EXIT_FAILURE;
  }
  if (block_shape.x > prop.maxThreadsDim[0] || block_shape.y > prop.maxThreadsDim[1] || block_shape.z > prop.maxThreadsDim[2]) {
    return EXIT_FAILURE;
  }
  auto threads_per_block = block_shape.x * block_shape.y * block_shape.z;
  if (threads_per_block > prop.maxThreadsPerBlock) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing check_block.cuh


In [23]:
%%writefile threads_host.cuh
#pragma once
#include <iostream>
#include <string>

__host__
inline void welcome() {
  std::string msg = "This program will launch a GPU kernel and print its thread number";
  std::cout << msg << std::endl;
}

Writing threads_host.cuh


In [24]:
%%writefile threads_device.cuh
#pragma once
#include <cstdint>

__device__
inline uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing threads_device.cuh


In [25]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdint>
#include <cstdio>
#include "threads_device.cuh"

__global__
static void kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [26]:
%%writefile run_kernel.cuh
#pragma once
#include <cuda_runtime.h>
#include "threads_kernel.cuh"

static inline cudaError_t run_kernel(const dim3& grid_shape, const dim3& block_shape) {
  kernel<<<grid_shape, block_shape>>>();
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }
  return cudaDeviceSynchronize();
}


Writing run_kernel.cuh


In [27]:
%%writefile threads.cu
#include <cstdint>
#include <cstdlib>
#include <format>
#include <iostream>
#include <span>
#include <string>
#include <cuda_runtime.h>
#include "check_block.cuh"
#include "check_grid.cuh"
#include "parse_num.hpp"
#include "run_kernel.cuh"
#include "threads_host.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);
  constexpr uint32_t device_id = 0;

  if (args.size() != 3) {
    std::string msg = std::format("Usage: {} <grid size> <block size>", args[0]);
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    std::string msg = std::format("Error setting device: {}", cudaGetErrorString(set_device_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  cudaDeviceProp prop{};
  auto prop_err = cudaGetDeviceProperties(&prop, device_id);
  if (prop_err != cudaSuccess) {
    std::string msg = std::format("Error getting device properties: {}", cudaGetErrorString(prop_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  uint32_t grid_size{};
  auto grid_size_err = parse_num(grid_size, args[1]);
  if (grid_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid size" << std::endl;
    std::exit(EXIT_FAILURE);
  }
  auto grid_shape = dim3{grid_size, 1, 1};
  auto grid_shape_err = check_grid(grid_shape, prop);
  if (grid_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid shape" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  uint32_t block_size{};
  auto block_size_err = parse_num(block_size, args[2]);
  if (block_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block size" << std::endl;
    std::exit(EXIT_FAILURE);
  }
  auto block_shape = dim3{block_size, 1, 1};
  auto block_shape_err = check_block(block_shape, prop);
  if (block_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block shape" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  welcome();
  auto kernel_err = run_kernel(grid_shape, block_shape);
  if (kernel_err != cudaSuccess) {
    std::string msg = std::format("Kernel failed: {}", cudaGetErrorString(kernel_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  return EXIT_SUCCESS;
}


Writing threads.cu


In [28]:
%%bash
nvcc -std=c++20 -arch=native -O3 "threads.cu" -o threads

In [29]:
%%bash
./threads 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 102: Extending Python with C

In [30]:
%%writefile lib_cuda.cu
#include <cstddef>
#include <cstdint>
#include <cuda_runtime.h>
#include "check_block.cuh"
#include "check_grid.cuh"
#include "run_kernel.cuh"
#include "threads_host.cuh"

extern "C" {
  cudaError_t c_set_device(uint32_t device_id) {
    return cudaSetDevice(device_id);
  }

  void c_welcome() {
    welcome();
  }

  size_t c_prop_size() {
    return sizeof(cudaDeviceProp);
  }

  cudaError_t c_get_prop(cudaDeviceProp* prop, uint32_t device_id) {
    return cudaGetDeviceProperties(prop, device_id);
  }

  int c_check_grid(dim3 grid_shape, const cudaDeviceProp* prop) {
    return check_grid(grid_shape, *prop);
  }

  int c_check_block(dim3 block_shape, const cudaDeviceProp* prop) {
    return check_block(block_shape, *prop);
  }

  cudaError_t c_run_kernel(dim3 grid_shape, dim3 block_shape) {
    return run_kernel(grid_shape, block_shape);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}


Writing lib_cuda.cu


In [31]:
%%bash
nvcc -std=c++20 -arch=native -O3 --compiler-options '-fPIC' -shared lib_cuda.cu -o lib_cuda.so

In [32]:
%%writefile mod_cuda.py
from ctypes import (
    CDLL,
    POINTER,
    Structure,
    byref,
    c_byte,
    c_char_p,
    c_int,
    c_size_t,
    c_uint32,
)
from pathlib import Path

class Dim3(Structure):
    _fields_ = [
        ("x", c_uint32),
        ("y", c_uint32),
        ("z", c_uint32)
    ]

    def __init__(self, x: int = 1, y: int = 1, z: int = 1):
        super().__init__(x, y, z)

    def __repr__(self) -> str:
        return f"Dim3({self.x}, {self.y}, {self.z})"

try:
    lib = CDLL(str(Path(__file__).parent / 'lib_cuda.so'))
except OSError as e:
    raise RuntimeError("Failed to load lib_cuda.so") from e

lib.c_prop_size.argtypes = []
lib.c_prop_size.restype  = c_size_t

class DeviceProp(Structure):
    _fields_ = [
        ("data", c_byte * lib.c_prop_size())
    ]

lib.c_set_device.argtypes = [c_uint32]
lib.c_set_device.restype  = c_int

lib.c_welcome.argtypes = []
lib.c_welcome.restype  = None

lib.c_get_prop.argtypes = [POINTER(DeviceProp), c_uint32]
lib.c_get_prop.restype  = c_int

lib.c_check_grid.argtypes = [Dim3, POINTER(DeviceProp)]
lib.c_check_grid.restype  = c_int

lib.c_check_block.argtypes = [Dim3, POINTER(DeviceProp)]
lib.c_check_block.restype  = c_int

lib.c_run_kernel.argtypes = [Dim3, Dim3]
lib.c_run_kernel.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def set_device(device_id: int) -> None:
    if (err := lib.c_set_device(device_id)) != 0:
        raise RuntimeError(f"Error setting device: {get_error_string(err)}")

def welcome() -> None:
    lib.c_welcome()

def get_prop(device_id: int) -> DeviceProp:
    prop = DeviceProp()
    if (err := lib.c_get_prop(byref(prop), device_id)) != 0:
        raise RuntimeError(f"Error getting device properties: {get_error_string(err)}")
    return prop

def check_grid(grid_shape: Dim3, prop: DeviceProp) -> None:
    if lib.c_check_grid(grid_shape, byref(prop)) != 0:
        raise ValueError("Invalid grid shape")

def check_block(block_shape: Dim3, prop: DeviceProp) -> None:
    if lib.c_check_block(block_shape, byref(prop)) != 0:
        raise ValueError("Invalid block shape")

def run_kernel(grid_shape: Dim3, block_shape: Dim3) -> None:
    if (err := lib.c_run_kernel(grid_shape, block_shape)) != 0:
        raise RuntimeError(f"Kernel failed: {get_error_string(err)}")


Writing mod_cuda.py


In [33]:
%%writefile threads.py
import sys

from mod_cuda import (
    Dim3,
    check_block,
    check_grid,
    get_prop,
    run_kernel,
    set_device,
    welcome,
)

if __name__ == '__main__':
    device_id = 0

    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <grid size> <block size>", file=sys.stderr)
        sys.exit(1)

    set_device(device_id)
    prop = get_prop(device_id)

    grid_size = int(sys.argv[1])
    grid_shape = Dim3(grid_size, 1, 1)
    check_grid(grid_shape, prop)

    block_size = int(sys.argv[2])
    block_shape = Dim3(block_size, 1, 1)
    check_block(block_shape, prop)

    welcome()
    run_kernel(grid_shape, block_shape)


Writing threads.py


In [34]:
%%bash
python3 threads.py 2 3


This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!
